旧版 Keras = 只能绑定 TensorFlow
以前的 Keras 默认只能用 TensorFlow 做后端，所以你不装 tensorflow 就报错。

新版 Keras 3.0+ = 支持多后端！
Keras 3.x 支持 3 种后端：
tensorflow（默认）
torch
jax

In [21]:
import os
os.environ["KERAS_BACKEND"] = "torch"  # 让 Keras 用 PyTorch 运行

import torch
import keras
from keras import layers

# 检查是否成功
print("keras:", keras.__version__)
print("torch:", torch.__version__)
print("backend:", keras.backend.backend())  # 输出 torch 就对了


keras: 3.10.0
torch: 2.8.0+cu128
backend: torch


In [22]:
import torch

x = torch.tensor([1.0, 2.0, 3.0])
print(x)
print(x * 2)

tensor([1., 2., 3.])
tensor([2., 4., 6.])


In [23]:
import torch

x = torch.tensor([1.0, 2.0, 3.0])
print(x)
print(x * 2)

tensor([1., 2., 3.])
tensor([2., 4., 6.])


In [24]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
from keras import layers

model = keras.Sequential([
    layers.Dense(8, activation="relu"),
    layers.Dense(1)
])

model.build((None, 4))
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 8)              │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 49 (196.00 B)

 Trainable params: 49 (196.00 B)

 Non-trainable params: 0 (0.00 B)

In [25]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
import numpy as np
from keras import layers

model = keras.Sequential([
    layers.Dense(8, activation="relu"),
    layers.Dense(1)
])

model.build((None, 4))

x = np.array([[1.0, 2.0, 3.0, 4.0]])
y = model(x)

print(y)

tensor([[0.3558]], device='cuda:0', grad_fn=<AddBackward0>)


In [26]:
from keras.datasets import mnist

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

In [27]:
train_images.shape

(60000, 28, 28)

In [28]:
len(train_labels)

60000

In [29]:
train_labels

array([5, 0, 4, ..., 5, 6, 8], dtype=uint8)

In [30]:
test_images.shape

(10000, 28, 28)

In [31]:
len(test_labels)

10000

In [32]:
test_labels

array([7, 2, 1, ..., 4, 5, 6], dtype=uint8)

In [33]:
from keras import models
from keras import layers

network = models.Sequential()
network.add(layers.Dense(512, activation='relu', input_shape=(28 * 28,)))
network.add(layers.Dense(10, activation='softmax'))

In [34]:
network.compile(optimizer='rmsprop',
                loss='categorical_crossentropy',
                metrics=['accuracy'])

In [35]:
train_images = train_images.reshape((60000, 28 * 28))
train_images = train_images.astype('float32') / 255

test_images = test_images.reshape((10000, 28 * 28))
test_images = test_images.astype('float32') / 255

In [36]:
from keras.utils import to_categorical

train_labels = to_categorical(train_labels)
test_labels = to_categorical(test_labels)

In [37]:
network.fit(train_images, train_labels, epochs=5, batch_size=128)

Epoch 1/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8725 - loss: 0.4364
Epoch 2/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9657 - loss: 0.1147
Epoch 3/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9784 - loss: 0.0720
Epoch 4/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9855 - loss: 0.0481
Epoch 5/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9893 - loss: 0.0375


In [38]:
test_loss, test_acc = network.evaluate(test_images, test_labels)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9757 - loss: 0.0738


In [40]:
print('test_acc:', test_acc)

test_acc: 0.9789999723434448


加层、加 Dropout（防止过拟合），增加训练轮数（epochs=12），换优化器rmsprop为adam

In [41]:
import os
os.environ["KERAS_BACKEND"] = "torch"

from keras import models, layers
from keras.datasets import mnist
from keras.utils import to_categorical

# 加载数据
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

# 预处理
train_images = train_images.reshape((60000, 28*28)).astype("float32") / 255
test_images = test_images.reshape((10000, 28*28)).astype("float32") / 255

train_labels = to_categorical(train_labels)
test_labels = to_categorical(test_labels)

# ✨ 超强改进版模型
model = models.Sequential()
model.add(layers.Dense(512, activation="relu", input_shape=(28*28,)))
model.add(layers.Dropout(0.2))
model.add(layers.Dense(256, activation="relu"))
model.add(layers.Dropout(0.2))
model.add(layers.Dense(128, activation="relu"))
model.add(layers.Dense(10, activation="softmax"))

model.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])

# 训练
model.fit(train_images, train_labels, epochs=12, batch_size=128)

# 测试
test_loss, test_acc = model.evaluate(test_images, test_labels)
print("最终准确率：", test_acc)

Epoch 1/12
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8473 - loss: 0.5025
Epoch 2/12
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9638 - loss: 0.1162
Epoch 3/12
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9742 - loss: 0.0841
Epoch 4/12
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9800 - loss: 0.0616
Epoch 5/12
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9843 - loss: 0.0494
Epoch 6/12
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9852 - loss: 0.0473
Epoch 7/12
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9872 - loss: 0.0400
Epoch 8/12
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9877 - loss: 0.0362
Epoch 9/12
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9883 - loss: 0.0341
Epoch 10/12
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9906 - loss: 0.0283
Epoch 11/12
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9898 - loss: 0.0300
Epoch 12/12
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step

换CNN（卷积神经网络）

In [42]:
import os
os.environ["KERAS_BACKEND"] = "torch"

from keras import models, layers
from keras.datasets import mnist
from keras.utils import to_categorical

# 加载数据
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

# CNN 不需要展平！保留 28x28 结构！
train_images = train_images.reshape((60000, 28, 28, 1)).astype("float32") / 255
test_images = test_images.reshape((10000, 28, 28, 1)).astype("float32") / 255

train_labels = to_categorical(train_labels)
test_labels = to_categorical(test_labels)

# ========================
# ✅ CNN 模型（图像神器）
# ========================
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation="relu", input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation="relu"),
    
    layers.Flatten(),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# 训练
model.fit(train_images, train_labels, epochs=12, batch_size=128)

# 测试
test_loss, test_acc = model.evaluate(test_images, test_labels)
print("最终 CNN 准确率：", test_acc)

Epoch 1/12


d:\Anaconda\envs\pytorch\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8112 - loss: 0.5860
Epoch 2/12
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9793 - loss: 0.0713
Epoch 3/12
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9838 - loss: 0.0516
Epoch 4/12
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9870 - loss: 0.0408
Epoch 5/12
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9887 - loss: 0.0361
Epoch 6/12
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9902 - loss: 0.0288
Epoch 7/12
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9917 - loss: 0.0256
Epoch 8/12
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9926 - loss: 0.0234
Epoch 9/12
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9921 - loss: 0.0238
Epoch 10/12
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9945 - loss: 0.0184
Epoch 11/12
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9944 - loss: 0.0182
Epoch 12/12
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy